### **HW-1 Feature Engineering Competition — Bank Marketing**

A Portuguese bank ran a direct-marketing campaign. Their callers contacted thousands of customers and offered them a term deposit. Your task: **predict which customers will say yes.**

#### **What this practice teaches**

Most machine learning courses give the impression that the algorithm is the interesting part. It usually isn't. In real projects, the model is something you pick off the shelf in 30 seconds; the **months of work** go into deciding what data to give it.

In this competition, the algorithm is **frozen**. Every team uses the same Random Forest with the same hyperparameters and the same random seed. The only thing you can change is how you transform the raw columns into features the model sees.

Whoever wins, wins because they understood the **business** better — not because they had a better algorithm.

#### **The rules**

1. **You cannot change the model.** A `RandomForestClassifier` with fixed hyperparameters and seed is provided in section 4. Do not touch it.
2. **You cannot change the train/test split.** It is set with a fixed seed.
3. **You can change one thing:** the `build_features()` function in section 5. Inside it, anything goes — create, drop, combine, encode, bucket, transform.
4. **No external data.** Only the columns shipped with the dataset.
5. **No data leakage.** You may not use the target `y` inside `build_features()`. You may not "peek" at the test set when computing aggregates.
6. **Submission:** save this notebook and a short write-up (see section 8).

The leaderboard ranks by **AUC** on the held-out test set.


#### **Run this notebook in COLAB**

<a target="_blank" href="https://colab.research.google.com/github/castorgit/RL_course/blob/main/00_LunarLander-COLAB_render.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---
#### **Setup — do not modify**
---
Loads the dataset, fixes the random seed, prepares the libraries we need. Run this cell as-is.

If your laptop has no internet access, ask your instructor for the CSV file `bank-additional-full.csv` and place it in the same folder as this notebook — the loader will pick it up automatically.


In [2]:
import io
import os
import zipfile
import urllib.request

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

In [3]:
# ------------------------------------------------------------------
# Reproducibility. Locked so every team trains on identical data.
# DO NOT CHANGE THIS SEED.
# ------------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [11]:
# Use a permissive SSL context to work around the expired UCI certificate.
# Safe here only because the URL is fixed and the data file is well-known.
import ssl
import urllib.request

LOCAL_ZIP = "bank-additional.zip"

request = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
insecure_ctx = ssl.create_default_context()
insecure_ctx.check_hostname = False
insecure_ctx.verify_mode = ssl.CERT_NONE

with urllib.request.urlopen(request, timeout=60, context=insecure_ctx) as response:
        zip_bytes = response.read()

# Save the zip locally so subsequent runs do not hit the network.
with open(LOCAL_ZIP, "wb") as f:
    f.write(zip_bytes)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
    with zf.open("bank-additional/bank-additional-full.csv") as f:
        df = pd.read_csv(f, sep=";")

## 2. The dataset

Each row represents one customer that the bank called during the campaign. The target column `y` is `"yes"` if the customer subscribed to a term deposit after the call, `"no"` otherwise.

### Columns

| Column | Meaning | Type |
|---|---|---|
| `age` | Customer age | numeric |
| `job` | Job category | categorical |
| `marital` | Marital status | categorical |
| `education` | Education level | categorical |
| `default` | Has credit currently in default? | categorical (yes / no / unknown) |
| `housing` | Has a housing loan? | categorical |
| `loan` | Has a personal loan? | categorical |
| `contact` | Communication channel used | categorical (cellular / telephone) |
| `month` | Month of the last contact | categorical |
| `day_of_week` | Day of week of the last contact | categorical |
| `duration` | Length of the last call in seconds | numeric — **see warning below** |
| `campaign` | Number of contacts during this campaign | numeric |
| `pdays` | Days since last contact in a previous campaign — `999` means "never" | numeric |
| `previous` | Contacts before this campaign | numeric |
| `poutcome` | Outcome of the previous campaign | categorical (success / failure / nonexistent) |
| `emp.var.rate` | Employment variation rate (macroeconomic) | numeric |
| `cons.price.idx` | Consumer price index | numeric |
| `cons.conf.idx` | Consumer confidence index | numeric |
| `euribor3m` | 3-month Euribor interest rate | numeric |
| `nr.employed` | Number of employees (macroeconomic) | numeric |
| `y` | **Target** — did the customer subscribe? | binary |

### ⚠️ Trap: the `duration` column

`duration` is the length of the call. It is only known **after** the call has happened. A model that uses it would be cheating — in production you'd have to decide who to call **before** the call. The UCI documentation says explicitly that this column should be dropped if you want a realistic model.

We drop it for you below. **Do not put it back.**


In [13]:
# Drop duration to prevent the most common form of leakage in this dataset.
# (See section 2 for the explanation.) DO NOT add it back inside build_features.
df = df.drop(columns=["duration"])

# Encode the target as 0/1 so AUC and sklearn metrics work cleanly.
df["y"] = (df["y"] == "yes").astype(int)

# Look at the class balance. The dataset is imbalanced — only about 11% of
# customers subscribe. This is why we use AUC (not accuracy) as the metric:
# a trivial model that always predicts "no" would already score 89% accuracy.
print("Class distribution:")
print(df["y"].value_counts(normalize=True).round(3).to_string())


Class distribution:
y
0    0.887
1    0.113


## 3. Train / test split — do not modify

Every team uses the same split with the same seed. Anything you do in `build_features()` will be evaluated on the same held-out test rows.

Note the `stratify=df["y"]` argument — it makes sure both halves have the same proportion of "yes" customers. With imbalanced data, an unstratified split can give you misleadingly easy or hard test sets.


In [14]:
train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df["y"],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")
print(f"Train positives: {train_df['y'].mean():.3f}")
print(f"Test  positives: {test_df['y'].mean():.3f}")


Train rows: 30,891
Test rows:  10,297
Train positives: 0.113
Test  positives: 0.113


## 4. The model — do not modify

A `RandomForestClassifier` with **fixed** hyperparameters and seed. You may not change anything in this cell. The leaderboard depends on every team running the same model.

A quick note on why these choices:

- `n_estimators=200` — enough trees to give stable predictions without making training slow.
- `max_depth=10` — limits each tree so the forest doesn't memorize the training set.
- `min_samples_leaf=20` — every leaf must cover at least 20 customers; prevents brittle rules.
- `class_weight="balanced"` — accounts for the 11% / 89% class imbalance.
- `random_state=RANDOM_STATE` — same trees for everyone.

If your features are good, this model will score well. If they are not, it won't — no matter how much you tweak.


In [15]:
# ==================================================
#   LOCKED CLASSIFIER — DO NOT MODIFY THIS CELL.
# ==================================================
def make_model() -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
# ==================================================


## 5. Your work goes here — `build_features()`

This is the only function you may edit. Read the rules below carefully.

### What you can do
- Create new columns from existing ones — ratios, differences, interaction terms, indicators.
- Bucket continuous variables (e.g. age ranges).
- Encode categorical variables — one-hot is provided as a baseline; you can try other encodings if you want.
- Drop columns you believe are noise.
- Re-scale, transform, or clean any column.

### What you cannot do
- Use the `y` column inside this function.
- Use any data not present in the input dataframe.
- Compute statistics on the test set. Anything that learns from data must come from the training set and be passed forward via `train_stats`.

### Why the `train_stats` argument exists

Some operations need to "remember" something from the training set so the test set can use the same recipe. The classic case is one-hot encoding: if a category appears only in test, you don't want a brand-new column to appear. The starter code shows the pattern — compute the feature list on the train call, reuse it on the test call.

If you invent a feature like "average campaign length for this job", compute it on the **training rows** only, store the resulting lookup table in `train_stats`, and merge it back on the test rows. The graders will check for this.


### Hints — kinds of features that often help in marketing data

Read these before you start. They are *categories of thinking*, not solutions.

**Time and seasonality.** `month` and `day_of_week` are strings. A model can't tell that December is closer to January than to June. Cyclic encodings (sin / cos of the position in the year, or position in the week) sometimes help. Holiday months may behave differently than ordinary ones.

**Lifecycle of contact.** A customer with `previous = 0` is brand new to the campaign. A customer with `previous = 5` has been called five times and is in a very different state of mind. Indicators like "first contact ever" or "has been contacted many times this campaign" can capture nonlinear behavior.

**Special values disguised as numbers.** The value `999` in `pdays` is a code for "never contacted before", not an actual count of days. A tree will treat it as just a very large number. Replacing it with `-1` or splitting it into a "was contacted before" flag plus a real day count often helps.

**Combining categoricals.** A young blue-collar worker and an old blue-collar worker are different leads. Interactions like `age_bucket × job` or `education × marital` can capture these segments without forcing the tree to find them on its own.

**Macroeconomic context.** `emp.var.rate`, `cons.conf.idx`, `euribor3m` change slowly over months. They may already explain the month effect — or they may complement it. Consider whether your engineered month features and the macro variables are redundant.

**Risk profile.** Customers with `default = "yes"`, `housing = "yes"`, and `loan = "yes"` together look very different from those with all "no". A single "loan burden" feature combining these may help the tree.

**Frequency / popularity.** Some job categories are very common, others are rare. Frequency encodings (count of occurrences per category) can give the tree a hint about which segments are statistically reliable.

You will not need all of these. **One thoughtful feature is worth ten arbitrary ones.** Pick a hypothesis, implement it cleanly, and check whether AUC goes up. If it doesn't, ask why before adding the next one.


In [16]:
# =================================================================
#   EDIT THIS FUNCTION. Everything else in the notebook is locked.
# =================================================================
def build_features(df_in: pd.DataFrame, train_stats: dict | None = None):
    """
    Transform raw columns into model-ready numeric features.

    Parameters
    ----------
    df_in : pd.DataFrame
        Raw input. The 'y' column may or may not be present; we drop it just
        in case to make sure we never accidentally feed it to the model.
    train_stats : dict or None
        On the first call (training), pass None and this function returns the
        stats it computed. On the second call (test), pass the dict you got
        back so train and test use the same recipe.

    Returns
    -------
    X : pd.DataFrame   Model-ready numeric features.
    train_stats : dict Stats / column list to reuse on the test set.
    """
    # Always work on a copy. Never mutate the input dataframe in place.
    df = df_in.drop(columns=["y"], errors="ignore").copy()

    # -------------------------------------------------------------
    # Tiny baseline so the notebook runs end-to-end out of the box.
    # Read these to learn the pattern, then replace / extend them
    # with your own ideas.
    # -------------------------------------------------------------

    # 1) Replace the "never contacted" sentinel in pdays.
    #    The tree treats 999 as a very large number, which is misleading.
    df["was_contacted_before"] = (df["pdays"] != 999).astype(int)
    df["pdays"] = df["pdays"].replace(999, -1)

    # 2) An obvious binary signal: a previous campaign that worked is a
    #    strong predictor of conversion. The model could find this on its
    #    own, but giving it the column makes the tree shallower and faster.
    df["prev_success"] = (df["poutcome"] == "success").astype(int)

    # -------------------------------------------------------------
    # >>> YOUR FEATURES GO HERE <<<
    # Add columns to `df` using the raw fields. A couple of examples
    # to spark ideas (commented out — uncomment, adapt, or replace):
    #
    # df["age_bucket"] = pd.cut(df["age"],
    #                           bins=[0, 25, 35, 50, 65, 120],
    #                           labels=["young", "early_career",
    #                                   "mid_career", "pre_retirement",
    #                                   "retired"])
    #
    # df["heavy_campaign"] = (df["campaign"] >= 5).astype(int)
    # -------------------------------------------------------------

    # -------------------------------------------------------------
    # Encode the remaining categorical columns.
    # We use one-hot with explicit columns so the train and test
    # dataframes end up with exactly the same shape.
    # -------------------------------------------------------------
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

    X = pd.get_dummies(df, columns=cat_cols, drop_first=False)

    if train_stats is None:
        # First call (training): remember which columns we produced.
        train_stats = {"feature_columns": X.columns.tolist()}
    else:
        # Second call (test): align to training columns. Any new categories
        # that only appear in test become all-zero columns; any training
        # columns missing in test are filled with zero.
        X = X.reindex(columns=train_stats["feature_columns"], fill_value=0)

    # Make sure everything is numeric (sklearn does not always accept booleans).
    return X.astype(float), train_stats


## 6. Train and score — do not modify

This cell calls your `build_features`, fits the locked model, and reports the AUC on the held-out test set. **This number is your leaderboard score.**

If something looks suspicious — for example, AUC = 0.95 on the first try — re-read the leakage warnings in sections 2 and 5. Scores above roughly 0.80 on this dataset (without `duration`) suggest you may be leaking the target somewhere.


In [17]:
# Build features on train, then on test using the same stats.
X_train, stats = build_features(train_df, train_stats=None)
X_test,  _     = build_features(test_df,  train_stats=stats)

y_train = train_df["y"].values
y_test  = test_df["y"].values

# Self-audit. Catches the most common mistakes before they bite.
assert X_train.shape[1] == X_test.shape[1], (
    "Train and test feature counts differ. "
    "Likely cause: a category appears only in train or only in test. "
    "Make sure your encoding uses the saved column list."
)
assert X_train.shape[0] == len(train_df), "Lost rows on the training side."
assert X_test.shape[0]  == len(test_df),  "Lost rows on the test side."
assert not X_train.isna().any().any(), "NaNs in training features — fix them in build_features."
assert not X_test.isna().any().any(),  "NaNs in test features — fix them in build_features."

print(f"Number of features: {X_train.shape[1]}")

# Fit the locked model.
model = make_model()
model.fit(X_train, y_train)

# Score with probabilities (AUC needs scores, not labels).
proba_test = model.predict_proba(X_test)[:, 1]
pred_test  = (proba_test >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba_test)
acc = accuracy_score(y_test, pred_test)

print()
print("================ LEADERBOARD SCORE ================")
print(f"Test AUC      : {auc:.4f}    <-- this is what we rank on")
print(f"Test Accuracy : {acc:.4f}")
print("===================================================")
print()
print("Confusion matrix (rows: true, cols: predicted):")
print(confusion_matrix(y_test, pred_test))
print()
print(classification_report(y_test, pred_test, target_names=["no", "yes"]))


Number of features: 64

================ LEADERBOARD SCORE ================
Test AUC      : 0.8138    <-- this is what we rank on
Test Accuracy : 0.8535

Confusion matrix (rows: true, cols: predicted):
[[8046 1091]
 [ 418  742]]

              precision    recall  f1-score   support

          no       0.95      0.88      0.91      9137
         yes       0.40      0.64      0.50      1160

    accuracy                           0.85     10297
   macro avg       0.68      0.76      0.71     10297
weighted avg       0.89      0.85      0.87     10297



## 7. Inspect what the model learned

Feature importances tell you which columns the forest relied on most. Use this cell as your debugging tool:

- If a feature you spent hours on shows up near the bottom, ask whether it actually captures the business signal you intended.
- If a feature you didn't expect dominates, ask whether it might be leaking the target.
- If two of your engineered features are very correlated, the importance gets split between them — high importance for both is not as good as it looks.

Importance is suggestive, not definitive. The cleanest signal is still: did the AUC go up?


In [18]:
importance = pd.DataFrame({
    "feature":    X_train.columns,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Top 20 features by importance:")
print(importance.head(20).to_string(index=False))


Top 20 features by importance:
             feature  importance
         nr.employed    0.171516
           euribor3m    0.165444
        emp.var.rate    0.129614
       cons.conf.idx    0.075488
      cons.price.idx    0.047414
               pdays    0.042289
was_contacted_before    0.035502
        prev_success    0.035443
                 age    0.028706
    poutcome_success    0.027804
    contact_cellular    0.024581
   contact_telephone    0.023322
           month_may    0.022874
            campaign    0.014395
            previous    0.012759
           month_oct    0.010753
poutcome_nonexistent    0.009975
     default_unknown    0.009184
          default_no    0.008954
           month_mar    0.008518


## 8. Submission

Before submitting, walk through this checklist:

1. **Did you change any cell other than `build_features`?** If yes, your run is disqualified. The model, the split, and the scoring cell must be untouched.
2. **Is `duration` in your feature set?** Run `"duration" in X_train.columns`. The answer must be `False`.
3. **Did you compute any statistic on the test set?** Anything that learns from data must live inside `build_features` and use only the input dataframe.
4. **Does your write-up explain your features?** A list of column names is not enough.

### What to submit

- This notebook, saved with all cells executed.
- A short PDF or markdown write-up with:
  1. A numbered list of the features you created.
  2. The **business reasoning** for each one (one or two sentences).
  3. Your final AUC.
  4. One feature you tried that did **not** help, with a hypothesis for why.

A team that creates four well-reasoned features and writes a clear explanation will score higher on the rubric than a team that throws thirty arbitrary transformations at the model — even if the latter has a marginally higher AUC.
